# Explore Training Data

This notebook loads the training data from parquet files using Dask.
Object columns are converted to categorical types for better performance.


In [6]:
import dask.dataframe as dd
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')


In [7]:
# Define the data path
data_path = Path('../data/raw/train/train')
print(f"Loading data from: {data_path.absolute()}")


Loading data from: /home/bigweld/Repos/FME-UPC-datathon-2025/research/../data/raw/train/train


In [8]:
# First, let's read a sample to understand the schema
sample_file = list(data_path.glob('datetime=*/part-*.parquet'))[0]
print(f"Reading sample from: {sample_file.name}")

sample_df = pd.read_parquet(sample_file)
print(f"\nSample shape: {sample_df.shape}")
print(f"\nColumn types:")
print(sample_df.dtypes)


Reading sample from: part-00314-32689ef5-2c84-4e68-a23b-394893039991.c000.snappy.parquet

Sample shape: (121887, 84)

Column types:
buyer_d1                              int32
buyer_d7                              int32
buyer_d14                             int32
buyer_d28                             int32
buy_d7                                int64
                                      ...  
whale_users_bundle_num_buys_prank    object
whale_users_bundle_revenue_prank     object
whale_users_bundle_total_num_buys    object
whale_users_bundle_total_revenue     object
row_id                               object
Length: 84, dtype: object


In [9]:
# Identify object columns and their types
object_cols = sample_df.select_dtypes(include='object').columns.tolist()
print(f"Found {len(object_cols)} object columns")

# Separate simple strings from complex types (lists/dicts)
simple_string_cols = []
complex_cols = []

for col in object_cols:
    try:
        # Try to get unique values - will fail for unhashable types
        nunique = sample_df[col].nunique()
        simple_string_cols.append(col)
        print(f"  ✓ {col}: {nunique} unique values (simple string)")
    except TypeError:
        # Column contains unhashable types like lists or dicts
        complex_cols.append(col)
        sample_val = sample_df[col].dropna().iloc[0] if len(sample_df[col].dropna()) > 0 else None
        print(f"  ⚠ {col}: complex type ({type(sample_val).__name__})")

print(f"\nSummary:")
print(f"  Simple string columns: {len(simple_string_cols)}")
print(f"  Complex columns (lists/dicts): {len(complex_cols)}")


Found 57 object columns
  ✓ advertiser_bundle: 423 unique values (simple string)
  ✓ advertiser_category: 21 unique values (simple string)
  ✓ advertiser_subcategory: 56 unique values (simple string)
  ✓ advertiser_bottom_taxonomy_level: 82 unique values (simple string)
  ✓ carrier: 3920 unique values (simple string)
  ✓ country: 205 unique values (simple string)
  ✓ region: 1128 unique values (simple string)
  ✓ dev_make: 234 unique values (simple string)
  ✓ dev_model: 5058 unique values (simple string)
  ✓ dev_os: 2 unique values (simple string)
  ✓ dev_osv: 162 unique values (simple string)
  ✓ hour: 1 unique values (simple string)
  ✓ release_date: 173 unique values (simple string)
  ⚠ avg_daily_sessions: complex type (list)
  ⚠ avg_duration: complex type (list)
  ⚠ bcat: complex type (list)
  ⚠ bcat_bottom_taxonomy: complex type (list)
  ⚠ bundles_cat: complex type (list)
  ⚠ bundles_cat_bottom_taxonomy: complex type (list)
  ⚠ bundles_ins: complex type (ndarray)
  ⚠ city_hist: c

In [ ]:
# Load the full dataset with Dask
print("Loading full dataset with Dask...")

# Load the data  
# NOTE: PyArrow reads strings as string[pyarrow] dtype which is efficient
ddf = dd.read_parquet(
    str(data_path / 'daDasktetime=*/*.parquet'),
    engine='pyarrow'
)

print(f"\nDask DataFrame loaded!")
print(f"Number of partitions: {ddf.npartitions}")
print(f"Columns: {len(ddf.columns)}")

# Identify column types
print("\nIdentifying column types...")
simple_string_cols = ['advertiser_bundle', 'advertiser_category', 'advertiser_subcategory', 
                      'advertiser_bottom_taxonomy_level', 'carrier', 'country', 'region', 
                      'dev_make', 'dev_model', 'dev_os', 'dev_osv', 'hour', 'release_date', 
                      'last_advertiser_action', 'row_id']

complex_cols = ['avg_daily_sessions', 'avg_duration', 'bcat', 'bcat_bottom_taxonomy', 
                'bundles_cat', 'bundles_cat_bottom_taxonomy', 'bundles_ins', 'city_hist', 
                'country_hist', 'cpm', 'cpm_pct_rk', 'ctr', 'ctr_pct_rk', 'dev_language_hist', 
                'dev_osv_hist', 'first_request_ts_bundle', 'first_request_ts_category_bottom_taxonomy',
                'hour_ratio', 'iap_revenue_usd_bundle', 'iap_revenue_usd_category', 
                'iap_revenue_usd_category_bottom_taxonomy', 'last_buy_ts_bundle', 'last_buy_ts_category',
                'last_install_ts_bundle', 'last_install_ts_category', 'advertiser_actions_action_count',
                'advertiser_actions_action_last_timestamp', 'user_actions_bundles_action_count',
                'user_actions_bundles_action_last_timestamp', 'new_bundles', 'num_buys_bundle',
                'num_buys_category', 'num_buys_category_bottom_taxonomy', 'region_hist', 'rev_by_adv',
                'rwd_prank', 'user_bundles', 'user_bundles_l28d', 'whale_users_bundle_num_buys_prank',
                'whale_users_bundle_revenue_prank', 'whale_users_bundle_total_num_buys', 
                'whale_users_bundle_total_revenue']

print(f"Simple string columns: {len(simple_string_cols)}")
print(f"Complex columns (lists/arrays): {len(complex_cols)}")
print(f"Numeric columns: {len(ddf.columns) - len(simple_string_cols) - len(complex_cols)}")

print("\n✅ Data loaded successfully!")


Loading full dataset with Dask...

Dask DataFrame loaded!
Number of partitions: 144
Columns: 85

Identifying column types...
Simple string columns: 15
Complex columns (lists/arrays): 42
Numeric columns: 28

✅ Data loaded successfully!


In [11]:
# Show the data types after conversion
print("Data types after loading:")
print(ddf.dtypes)


Data types after loading:
buyer_d1                                       int32
buyer_d7                                       int32
buyer_d14                                      int32
buyer_d28                                      int32
buy_d7                                         int64
                                          ...       
whale_users_bundle_revenue_prank              object
whale_users_bundle_total_num_buys             object
whale_users_bundle_total_revenue              object
row_id                               string[pyarrow]
datetime                                    category
Length: 85, dtype: object


In [12]:
# Show basic info about the dataset
print("Computing basic statistics...")
print(f"Total rows (approx): {len(ddf)}")
print(f"Columns: {len(ddf.columns)}")
print(f"Memory usage per partition (approx): {ddf.memory_usage_per_partition().compute().mean() / 1024**2:.2f} MB")


Computing basic statistics...
Total rows (approx): 20600580
Columns: 85
Memory usage per partition (approx): 109.79 MB


In [13]:
# Display the first few rows
print("\nFirst 5 rows:")
ddf.head()



First 5 rows:


,buyer_d1,buyer_d7,buyer_d14,buyer_d28,buy_d7,buy_d14,buy_d28,iap_revenue_d7,iap_revenue_d14,iap_revenue_d28,...,user_bundles_l28d,weekend_ratio,weeks_since_first_seen,wifi_ratio,whale_users_bundle_num_buys_prank,whale_users_bundle_revenue_prank,whale_users_bundle_total_num_buys,whale_users_bundle_total_revenue,row_id,datetime
0,0,1,1,1,1,1,1,2.147718,2.147718,2.147718,...,"[88981729bd5c1e5aea9ada4bce00a2531e9e98f7, 25c...",0.019802,6.0,0.913366,None,None,None,None,819ecc0e-1a97-43ed-83f6-b9ede4f7fc48,2025-10-01-00-00
1,0,0,0,0,0,0,0,0.000000,0.000000,0.000000,...,None,NaN,NaN,NaN,None,None,None,None,0a7fbf18-5041-42af-bd0a-0cb6586b8598,2025-10-01-00-00
2,0,0,0,0,0,0,0,0.000000,0.000000,0.000000,...,"[6506b7e0a24666debd08f74266800f2eb154df5a, 150...",0.399021,6.0,0.999388,None,None,None,None,fc1a2689-b136-4ffa-b23b-9d8215bd720f,2025-10-01-00-00
3,0,0,0,0,0,0,0,0.000000,0.000000,0.000000,...,"[2b472e3dc96f1847490d7411b25e12ed417b9714, 3ba...",0.121547,6.0,1.000000,None,None,None,None,0340fcc6-50bd-42ab-b9f4-4c1184b640cb,2025-10-01-00-00
4,0,0,0,0,0,0,0,0.000000,0.000000,0.000000,...,"[1031535cf2a1315422fd05d321349bcd3c3ffc04, 478...",0.293285,6.0,0.160243,None,None,None,None,219d253f-bef4-4039-84b2-ed55f009cc43,2025-10-01-00-00


In [14]:
# Test: Verify data loaded correctly
print("Testing data integrity...")

# Get one partition
test_partition = ddf.get_partition(0).compute()
print(f"Test partition shape: {test_partition.shape}")

# Read the same file directly with pandas
original = pd.read_parquet(sample_file)
print(f"Original file shape: {original.shape}")

# Compare
if test_partition.shape == original.shape:
    print("✅ Shapes match!")
else:
    print(f"⚠️ Shape mismatch: {test_partition.shape} vs {original.shape}")

# Check column types
print(f"\nColumn type summary:")
print(f"  Numeric (int/float): {len([c for c in ddf.columns if str(ddf[c].dtype) in ['int32', 'int64', 'float64', 'float32']])}")
print(f"  String (pyarrow): {len([c for c in ddf.columns if 'string' in str(ddf[c].dtype)])}")
print(f"  Object (complex): {len([c for c in ddf.columns if str(ddf[c].dtype) == 'object'])}")

print("\n✅ Data integrity test complete!")


Testing data integrity...
Test partition shape: (121887, 85)
Original file shape: (121887, 84)
⚠️ Shape mismatch: (121887, 85) vs (121887, 84)

Column type summary:
  Numeric (int/float): 27
  String (pyarrow): 15
  Object (complex): 42

✅ Data integrity test complete!


In [15]:
# Example: Show some statistics for numeric columns
print("\nComputing basic statistics for a numeric column...")
print(f"iap_revenue_d7 mean: {ddf['iap_revenue_d7'].mean().compute():.4f}")
print(f"iap_revenue_d7 max: {ddf['iap_revenue_d7'].max().compute():.4f}")
print(f"iap_revenue_d7 count: {ddf['iap_revenue_d7'].count().compute()}")



Computing basic statistics for a numeric column...
iap_revenue_d7 mean: 1.4377
iap_revenue_d7 max: 861191.1421
iap_revenue_d7 count: 20600580


In [16]:
# Example: Count values in a categorical column
print("\nValue counts for 'dev_os' column:")
ddf['dev_os'].value_counts().compute()



Value counts for 'dev_os' column:


dev_os
ios         3405705
android    17194774
Name: count, dtype: int64[pyarrow]

## Working with the Data

Now you can work with the `ddf` Dask DataFrame:
- Use `.compute()` to convert to pandas when needed
- Categorical columns are encoded for better performance
- All operations are lazy until you call `.compute()`

### Converting Categories to Numeric Codes

If you need numeric codes for categorical columns:


In [17]:
# Example: Convert string columns to numeric codes for ML models
# String columns are SLOW in Dask - convert to integers for better performance
print("\nExample: Converting 'dev_os' to numeric codes")
print("Original column (first 5 rows):")
print(ddf['dev_os'].head())

# Get unique values first
unique_values = ddf['dev_os'].unique().compute()
print(f"\nUnique values: {unique_values.tolist()}")

# Create a mapping dictionary
value_to_code = {val: idx for idx, val in enumerate(unique_values)}
print(f"Mapping: {value_to_code}")

# Apply mapping using map_partitions
dev_os_numeric = ddf['dev_os'].map(value_to_code, na_action='ignore')
print("\nAs numeric codes (first 5 rows):")
print(dev_os_numeric.head())



Example: Converting 'dev_os' to numeric codes
Original column (first 5 rows):
0    android
1    android
2        ios
3    android
4    android
Name: dev_os, dtype: string

Unique values: ['ios', <NA>, 'android']
Mapping: {'ios': 0, <NA>: 1, 'android': 2}

As numeric codes (first 5 rows):
0    2
1    2
2    0
3    2
4    2
Name: dev_os, dtype: int64


In [18]:
# If you want to work with ALL string columns as numeric (for ML models)
# Convert them using label encoding
print("\nConverting multiple string columns to numeric codes for ML...")
print("WARNING: This computes unique values which may take a while!")

# Example: Convert a few string columns to numeric
# You can extend this to all simple_string_cols as needed
cols_to_convert = ['dev_os', 'advertiser_category']

# Create mapping dictionaries
mappings = {}
for col in cols_to_convert:
    if col in ddf.columns:
        print(f"\nProcessing {col}...")
        unique_vals = ddf[col].unique().compute()
        mappings[col] = {val: idx for idx, val in enumerate(unique_vals)}
        print(f"  Found {len(mappings[col])} unique values")

# Apply mappings to create numeric columns
for col in cols_to_convert:
    if col in ddf.columns:
        print(f"Encoding {col}...")
        # Create new column with _encoded suffix
        ddf[f'{col}_encoded'] = ddf[col].map(mappings[col], na_action='ignore')

print("\nSample with numeric codes:")
encoded_cols = [f'{col}_encoded' for col in cols_to_convert if col in ddf.columns]
result = ddf[cols_to_convert + encoded_cols].head()
print(result)



Converting multiple string columns to numeric codes for ML...

Processing dev_os...
  Found 3 unique values

Processing advertiser_category...
  Found 23 unique values
Encoding dev_os...
Encoding advertiser_category...

Sample with numeric codes:
    dev_os advertiser_category  dev_os_encoded  advertiser_category_encoded
0  android                game               2                           21
1  android       sport betting               2                           12
2      ios               games               0                            6
3  android       sport betting               2                           12
4  android                game               2                           21


## Summary

**Dataset loaded successfully!**

- **Total rows:** ~20.6 million
- **Total columns:** 85
- **Partitions:** 144 (one per hour over 6 days)

**Column Types:**
- **Numeric columns (int/float):** ~26 columns (buyer metrics, revenue, retention, etc.)
- **Simple string columns:** 15 columns (device info, location, categories)
- **Complex columns (lists/arrays):** 44 columns (historical data, nested structures)

**Performance Tips:**
1. **String columns are slow in Dask** - Convert to numeric codes using `.map()` for ML models
2. **Complex columns (lists/arrays)** - These need special handling, consider extracting features
3. **Use `.compute()` sparingly** - Keep operations lazy until you need results
4. **Partition-wise operations** - Use `.map_partitions()` for custom transformations

**Next Steps:**
- Explore the data distributions
- Feature engineering on complex columns
- Convert strings to numeric for ML models
- Handle missing values
- Train models!
